# AutoGluon Image Classification on CIFAR-10

This notebook demonstrates using AutoGluon for image classification on the CIFAR-10 dataset.

**Features:**
- Uses all supported backbones from AutoGluon MultiModalPredictor
- Hyperparameter Optimization (HPO)
- Model leaderboard analysis
- Feature importance visualization
- Best model evaluation
- Model persistence

**Environment:** Modal.com notebook with A100 GPU

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install -q autogluon matplotlib seaborn pandas numpy scikit-learn pillow

In [ ]:
import os
import pickle
import tarfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path

from autogluon.multimodal import MultiModalPredictor
from autogluon.core.metrics import make_scorer

import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Setup complete!")

## 2. Download and Prepare CIFAR-10 Dataset

In [ ]:
# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Create directories
data_dir = Path('./cifar10_data')
data_dir.mkdir(exist_ok=True)

# Download CIFAR-10 dataset
cifar10_url = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
cifar10_file = data_dir / 'cifar-10-python.tar.gz'

if not cifar10_file.exists():
    print("Downloading CIFAR-10 dataset...")
    urllib.request.urlretrieve(cifar10_url, cifar10_file)
    print("Download complete!")
else:
    print("CIFAR-10 dataset already downloaded.")

# Extract dataset
extract_dir = data_dir / 'cifar-10-batches-py'
if not extract_dir.exists():
    print("Extracting dataset...")
    with tarfile.open(cifar10_file, 'r:gz') as tar:
        tar.extractall(path=data_dir)
    print("Extraction complete!")
else:
    print("Dataset already extracted.")

In [ ]:
def unpickle(file):
    """Load CIFAR-10 batch file"""
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

def save_cifar10_images(data_dir, output_dir, batch_files, prefix='train'):
    """Convert CIFAR-10 data to image files and create DataFrame"""
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)
    
    all_images = []
    all_labels = []
    all_filenames = []
    
    for batch_file in batch_files:
        batch_data = unpickle(data_dir / batch_file)
        images = batch_data[b'data']
        labels = batch_data[b'labels']
        
        # Reshape images from (10000, 3072) to (10000, 32, 32, 3)
        images = images.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        
        for i, (img, label) in enumerate(zip(images, labels)):
            # Save image
            img_filename = f"{prefix}_{len(all_images):05d}.png"
            img_path = output_path / img_filename
            Image.fromarray(img).save(img_path)
            
            all_images.append(str(img_path.absolute()))
            all_labels.append(class_names[label])
            all_filenames.append(img_filename)
    
    df = pd.DataFrame({
        'image': all_images,
        'label': all_labels,
        'filename': all_filenames
    })
    
    return df

print("Loading and converting CIFAR-10 data...")

# Process training data
train_batches = ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5']
train_df = save_cifar10_images(
    extract_dir, 
    data_dir / 'train_images', 
    train_batches, 
    prefix='train'
)

# Process test data
test_df = save_cifar10_images(
    extract_dir, 
    data_dir / 'test_images', 
    ['test_batch'], 
    prefix='test'
)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

# Display class distribution
print("\nTraining set class distribution:")
print(train_df['label'].value_counts().sort_index())

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample CIFAR-10 Images', fontsize=16, fontweight='bold')

for idx, class_name in enumerate(class_names):
    sample = train_df[train_df['label'] == class_name].iloc[0]
    img = Image.open(sample['image'])
    
    ax = axes[idx // 5, idx % 5]
    ax.imshow(img)
    ax.set_title(class_name, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Create Train/Validation Split

We'll use a portion of the training data for validation to monitor model performance during training.

In [ ]:
from sklearn.model_selection import train_test_split

# Split training data into train and validation sets
train_data, val_data = train_test_split(
    train_df, 
    test_size=0.1, 
    random_state=42, 
    stratify=train_df['label']
)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_df)}")

# Save splits to CSV for AutoGluon
train_data.to_csv(data_dir / 'train.csv', index=False)
val_data.to_csv(data_dir / 'val.csv', index=False)
test_df.to_csv(data_dir / 'test.csv', index=False)

print("\nData splits saved!")

## 4. List All Supported Backbones

In [ ]:
# List all supported pretrained models
supported_models = MultiModalPredictor.list_supported_models(pretrained=True)

print("All Supported Pretrained Models:")
print("=" * 50)
for i, model in enumerate(supported_models, 1):
    print(f"{i}. {model}")

print(f"\nTotal: {len(supported_models)} models")

## 5. Configure AutoGluon with HPO and All Backbones

We'll configure AutoGluon to:
- Use all supported pretrained backbones
- Enable Hyperparameter Optimization (HPO)
- Set appropriate time limits for training

In [ ]:
# Define model save path
model_path = './autogluon_cifar10_model'

# Define hyperparameter search space for HPO
hyperparameters = {
    'model.names': supported_models,  # Use all supported models
    'optimization.learning_rate': [1e-5, 5e-5, 1e-4, 5e-4],
    'optimization.max_epochs': [5, 10, 15],
    'optimization.warmup_steps': [0.1, 0.2],
    'env.batch_size': [32, 64, 128],
}

# Set hyperparameter tuning strategy
hyperparameter_tune_kwargs = {
    'searcher': 'random',  # Random search for HPO
    'scheduler': 'local',  # Local scheduler
    'num_trials': 20,  # Number of different hyperparameter configurations to try
}

print("Configuration complete!")
print(f"\nModels to train: {len(supported_models)}")
print(f"HPO trials per model: {hyperparameter_tune_kwargs['num_trials']}")

## 6. Train AutoGluon Model

This will train multiple models with different backbones and hyperparameters.
Training may take considerable time depending on the number of models and GPU availability.

In [ ]:
# Initialize the predictor
predictor = MultiModalPredictor(
    label='label',
    path=model_path,
    problem_type='multiclass',
    eval_metric='accuracy',
)

print("Starting model training...")
print("This may take a while depending on the number of models and dataset size.")
print("=" * 70)

# Train with HPO
predictor.fit(
    train_data=train_data,
    tuning_data=val_data,
    hyperparameters=hyperparameters,
    hyperparameter_tune_kwargs=hyperparameter_tune_kwargs,
    time_limit=7200,  # 2 hours time limit (adjust as needed)
    save_path=model_path,
)

print("\n" + "=" * 70)
print("Training complete!")

## 7. View and Analyze Model Leaderboard

The leaderboard shows all trained models ranked by their performance.

In [ ]:
# Get the leaderboard
leaderboard = predictor.leaderboard(data=val_data, silent=False)

print("\n" + "=" * 70)
print("MODEL LEADERBOARD")
print("=" * 70)
print(leaderboard.to_string())

# Save leaderboard to CSV
leaderboard.to_csv('./model_leaderboard.csv', index=False)
print("\nLeaderboard saved to 'model_leaderboard.csv'")

In [ ]:
# Visualize leaderboard
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('AutoGluon Model Leaderboard Analysis', fontsize=16, fontweight='bold')

# Plot 1: Top 10 models by score
ax1 = axes[0, 0]
top_models = leaderboard.head(10).copy()
top_models = top_models.sort_values('score_val', ascending=True)
ax1.barh(range(len(top_models)), top_models['score_val'], color='skyblue')
ax1.set_yticks(range(len(top_models)))
ax1.set_yticklabels(top_models['model'], fontsize=9)
ax1.set_xlabel('Validation Score', fontsize=12)
ax1.set_title('Top 10 Models by Validation Score', fontsize=12, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Plot 2: Training time comparison
ax2 = axes[0, 1]
top_models_time = leaderboard.head(10).copy()
ax2.bar(range(len(top_models_time)), top_models_time['fit_time'], color='coral')
ax2.set_xticks(range(len(top_models_time)))
ax2.set_xticklabels(range(1, len(top_models_time) + 1))
ax2.set_xlabel('Model Rank', fontsize=12)
ax2.set_ylabel('Training Time (seconds)', fontsize=12)
ax2.set_title('Training Time for Top 10 Models', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Plot 3: Score distribution
ax3 = axes[1, 0]
ax3.hist(leaderboard['score_val'], bins=20, color='lightgreen', edgecolor='black')
ax3.set_xlabel('Validation Score', fontsize=12)
ax3.set_ylabel('Frequency', fontsize=12)
ax3.set_title('Distribution of Model Scores', fontsize=12, fontweight='bold')
ax3.axvline(leaderboard['score_val'].mean(), color='red', linestyle='--', 
            linewidth=2, label=f"Mean: {leaderboard['score_val'].mean():.4f}")
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# Plot 4: Score vs Training Time scatter
ax4 = axes[1, 1]
scatter = ax4.scatter(leaderboard['fit_time'], leaderboard['score_val'], 
                     c=leaderboard['score_val'], cmap='viridis', 
                     s=100, alpha=0.6, edgecolors='black')
ax4.set_xlabel('Training Time (seconds)', fontsize=12)
ax4.set_ylabel('Validation Score', fontsize=12)
ax4.set_title('Score vs Training Time', fontsize=12, fontweight='bold')
ax4.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax4, label='Score')

plt.tight_layout()
plt.savefig('./leaderboard_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Leaderboard visualization saved to 'leaderboard_analysis.png'")

In [ ]:
# Summary statistics
print("\nLeaderboard Summary Statistics:")
print("=" * 70)
print(f"Total models trained: {len(leaderboard)}")
print(f"Best validation score: {leaderboard['score_val'].max():.4f}")
print(f"Worst validation score: {leaderboard['score_val'].min():.4f}")
print(f"Mean validation score: {leaderboard['score_val'].mean():.4f}")
print(f"Std validation score: {leaderboard['score_val'].std():.4f}")
print(f"\nTotal training time: {leaderboard['fit_time'].sum():.2f} seconds")
print(f"Average training time: {leaderboard['fit_time'].mean():.2f} seconds")
print(f"\nBest model: {leaderboard.iloc[0]['model']}")
print(f"Best model score: {leaderboard.iloc[0]['score_val']:.4f}")

## 8. Feature Importance Visualization

For image classification, feature importance shows which parts of the images contribute most to predictions.

In [ ]:
# Get feature importance
try:
    importance_df = predictor.feature_importance(data=val_data)
    
    if importance_df is not None and len(importance_df) > 0:
        print("Feature Importance:")
        print("=" * 70)
        print(importance_df.to_string())
        
        # Visualize feature importance
        plt.figure(figsize=(12, 6))
        importance_sorted = importance_df.sort_values('importance', ascending=True)
        plt.barh(importance_sorted['feature'], importance_sorted['importance'], color='steelblue')
        plt.xlabel('Importance', fontsize=12)
        plt.ylabel('Feature', fontsize=12)
        plt.title('Feature Importance Analysis', fontsize=14, fontweight='bold')
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig('./feature_importance.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("\nFeature importance visualization saved to 'feature_importance.png'")
    else:
        print("Note: Feature importance is not available for image classification models.")
        print("This is expected as deep learning models for images don't provide traditional feature importance.")
        
except Exception as e:
    print(f"Feature importance analysis not available for this model type: {e}")
    print("\nFor image models, we'll analyze prediction confidence instead.")

In [ ]:
# Alternative: Analyze prediction confidence as a proxy for feature importance
print("\nAnalyzing prediction confidence across classes...")

# Get predictions with probabilities
predictions = predictor.predict(val_data, as_pandas=True)
probabilities = predictor.predict_proba(val_data, as_pandas=True)

# Calculate confidence metrics
confidence_per_class = {}
for class_name in class_names:
    if class_name in probabilities.columns:
        class_mask = val_data['label'] == class_name
        class_probs = probabilities.loc[class_mask, class_name]
        confidence_per_class[class_name] = {
            'mean_confidence': class_probs.mean(),
            'std_confidence': class_probs.std(),
            'min_confidence': class_probs.min(),
            'max_confidence': class_probs.max()
        }

# Visualize confidence per class
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Prediction Confidence Analysis', fontsize=16, fontweight='bold')

# Mean confidence by class
ax1 = axes[0]
classes = list(confidence_per_class.keys())
mean_conf = [confidence_per_class[c]['mean_confidence'] for c in classes]
std_conf = [confidence_per_class[c]['std_confidence'] for c in classes]

ax1.bar(classes, mean_conf, color='lightblue', edgecolor='black', alpha=0.7)
ax1.errorbar(classes, mean_conf, yerr=std_conf, fmt='none', color='red', 
             capsize=5, capthick=2, label='Std Dev')
ax1.set_xlabel('Class', fontsize=12)
ax1.set_ylabel('Mean Confidence', fontsize=12)
ax1.set_title('Average Prediction Confidence by Class', fontsize=12, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Confidence distribution
ax2 = axes[1]
all_max_probs = probabilities.max(axis=1)
ax2.hist(all_max_probs, bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
ax2.set_xlabel('Maximum Prediction Probability', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Prediction Confidence', fontsize=12, fontweight='bold')
ax2.axvline(all_max_probs.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {all_max_probs.mean():.3f}')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('./prediction_confidence_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPrediction confidence analysis saved to 'prediction_confidence_analysis.png'")

## 9. Evaluate Best Model Performance

Comprehensive evaluation of the best model on the test set.

In [ ]:
# Evaluate on test data
print("Evaluating best model on test set...")
print("=" * 70)

test_score = predictor.evaluate(test_df, metrics=['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'])

print("\nTest Set Performance:")
print("=" * 70)
for metric, value in test_score.items():
    print(f"{metric}: {value:.4f}")

# Get predictions on test set
test_predictions = predictor.predict(test_df, as_pandas=True)
test_probabilities = predictor.predict_proba(test_df, as_pandas=True)

print("\nPredictions completed!")

In [ ]:
# Create confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

# Compute confusion matrix
cm = confusion_matrix(test_df['label'], test_predictions)

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix saved to 'confusion_matrix.png'")

In [ ]:
# Classification report
print("\nDetailed Classification Report:")
print("=" * 70)
print(classification_report(test_df['label'], test_predictions, 
                          target_names=class_names, digits=4))

# Save classification report
report_dict = classification_report(test_df['label'], test_predictions, 
                                   target_names=class_names, 
                                   output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv('./classification_report.csv')
print("\nClassification report saved to 'classification_report.csv'")

In [ ]:
# Per-class performance visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Per-Class Performance Metrics', fontsize=16, fontweight='bold')

metrics_data = report_df.iloc[:-3]  # Exclude avg rows

# Precision
ax1 = axes[0, 0]
ax1.bar(metrics_data.index, metrics_data['precision'], color='skyblue', edgecolor='black')
ax1.set_ylabel('Precision', fontsize=12)
ax1.set_title('Precision by Class', fontsize=12, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0, 1])

# Recall
ax2 = axes[0, 1]
ax2.bar(metrics_data.index, metrics_data['recall'], color='lightcoral', edgecolor='black')
ax2.set_ylabel('Recall', fontsize=12)
ax2.set_title('Recall by Class', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim([0, 1])

# F1-Score
ax3 = axes[1, 0]
ax3.bar(metrics_data.index, metrics_data['f1-score'], color='lightgreen', edgecolor='black')
ax3.set_ylabel('F1-Score', fontsize=12)
ax3.set_title('F1-Score by Class', fontsize=12, fontweight='bold')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(axis='y', alpha=0.3)
ax3.set_ylim([0, 1])

# Support
ax4 = axes[1, 1]
ax4.bar(metrics_data.index, metrics_data['support'], color='wheat', edgecolor='black')
ax4.set_ylabel('Support (# samples)', fontsize=12)
ax4.set_title('Number of Samples by Class', fontsize=12, fontweight='bold')
ax4.tick_params(axis='x', rotation=45)
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('./per_class_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("Per-class metrics visualization saved to 'per_class_metrics.png'")

In [ ]:
# Analyze misclassifications
misclassified = test_df.copy()
misclassified['predicted'] = test_predictions
misclassified['correct'] = misclassified['label'] == misclassified['predicted']

print(f"\nMisclassification Analysis:")
print("=" * 70)
print(f"Total test samples: {len(test_df)}")
print(f"Correctly classified: {misclassified['correct'].sum()} ({misclassified['correct'].mean()*100:.2f}%)")
print(f"Misclassified: {(~misclassified['correct']).sum()} ({(~misclassified['correct']).mean()*100:.2f}%)")

# Show some misclassified examples
misclassified_samples = misclassified[~misclassified['correct']].head(10)

if len(misclassified_samples) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle('Examples of Misclassified Images', fontsize=16, fontweight='bold')
    
    for idx, (_, sample) in enumerate(misclassified_samples.iterrows()):
        if idx >= 10:
            break
        img = Image.open(sample['image'])
        ax = axes[idx // 5, idx % 5]
        ax.imshow(img)
        ax.set_title(f"True: {sample['label']}\nPred: {sample['predicted']}", 
                    fontsize=9, color='red')
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('./misclassified_examples.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nMisclassified examples saved to 'misclassified_examples.png'")

## 10. Save the Model

The model has already been saved during training. Let's verify and document the saved artifacts.

In [ ]:
# Model is already saved during training
print("Model Save Location:")
print("=" * 70)
print(f"Path: {os.path.abspath(model_path)}")

# List contents of model directory
if os.path.exists(model_path):
    print("\nModel directory contents:")
    for root, dirs, files in os.walk(model_path):
        level = root.replace(model_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files[:10]:  # Limit to first 10 files per directory
            print(f"{subindent}{file}")
        if len(files) > 10:
            print(f"{subindent}... and {len(files) - 10} more files")
else:
    print("Model directory not found!")

print("\nModel successfully saved!")

In [ ]:
# Demonstrate loading the saved model
print("\nDemonstrating model loading...")
print("=" * 70)

# Load the model
loaded_predictor = MultiModalPredictor.load(model_path)

# Make a prediction with loaded model
sample_data = test_df.head(5)
loaded_predictions = loaded_predictor.predict(sample_data)

print("Model loaded successfully!")
print("\nSample predictions from loaded model:")
for idx, (_, row) in enumerate(sample_data.iterrows()):
    print(f"{idx+1}. True: {row['label']:12s} | Predicted: {loaded_predictions[idx]}")

print("\n" + "=" * 70)
print("Model persistence verified!")

## 11. Summary and Conclusion

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)

print("\n1. Dataset:")
print(f"   - Name: CIFAR-10")
print(f"   - Classes: {len(class_names)} ({', '.join(class_names)})")
print(f"   - Training samples: {len(train_data)}")
print(f"   - Validation samples: {len(val_data)}")
print(f"   - Test samples: {len(test_df)}")

print("\n2. Model Training:")
print(f"   - Framework: AutoGluon MultiModalPredictor")
print(f"   - Backbones tested: {len(supported_models)}")
print(f"   - HPO enabled: Yes")
print(f"   - HPO trials: {hyperparameter_tune_kwargs['num_trials']}")
print(f"   - Total models trained: {len(leaderboard)}")

print("\n3. Best Model Performance:")
best_model = leaderboard.iloc[0]
print(f"   - Model: {best_model['model']}")
print(f"   - Validation score: {best_model['score_val']:.4f}")
print(f"   - Training time: {best_model['fit_time']:.2f} seconds")

print("\n4. Test Set Performance:")
for metric, value in test_score.items():
    print(f"   - {metric}: {value:.4f}")

print("\n5. Outputs Generated:")
outputs = [
    'model_leaderboard.csv',
    'leaderboard_analysis.png',
    'prediction_confidence_analysis.png',
    'confusion_matrix.png',
    'classification_report.csv',
    'per_class_metrics.png',
    'misclassified_examples.png',
    f'{model_path}/ (model directory)'
]
for output in outputs:
    print(f"   - {output}")

print("\n" + "=" * 70)
print("EXPERIMENT COMPLETED SUCCESSFULLY!")
print("=" * 70)

print("\n📝 Next Steps:")
print("   1. Review the leaderboard to understand model performance")
print("   2. Analyze the confusion matrix for class-specific insights")
print("   3. Use the saved model for inference: MultiModalPredictor.load(model_path)")
print("   4. Fine-tune hyperparameters based on the analysis")
print("   5. Consider ensemble methods for improved performance")

## Notes for Modal.com Deployment

To run this notebook on Modal.com with A100 GPU:

1. **Environment Setup:**
   ```python
   import modal
   
   stub = modal.Stub("autogluon-cifar10")
   
   # Define image with AutoGluon and dependencies
   image = modal.Image.debian_slim().pip_install(
       "autogluon",
       "matplotlib",
       "seaborn",
       "pandas",
       "numpy",
       "scikit-learn",
       "pillow"
   )
   ```

2. **GPU Configuration:**
   ```python
   @stub.function(
       image=image,
       gpu="A100",  # Specify A100 GPU
       timeout=7200,  # 2 hours
       mounts=[modal.Mount.from_local_dir("./data", remote_path="/data")]
   )
   ```

3. **Resource Considerations:**
   - Training time depends on the number of models and HPO trials
   - Adjust `time_limit` and `num_trials` based on your budget
   - Monitor GPU memory usage for large backbones

4. **Data Persistence:**
   - Save important outputs to Modal volumes or external storage
   - Use Modal's persistent storage for model artifacts

For detailed Modal.com documentation, visit: https://modal.com/docs/guide/notebooks-modal